# machine-learning_004

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (119).ipynb`

| Field | Value |
|---|---|
| Section | `machine_learning` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 14 |
| Detected functions | calculate_indicators, apply_trading_strategy, train_ml_model, live_trade_decision, backtest_and_live_trade, calculate_win_rate, calculate_indicators, backtest_single_ticker, backtest_strategy, run_backtest_for_params |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/filtered_data.csv')
data

In [ ]:
data = data[data['Ticker'] == 'ICICIBANK']
data

In [ ]:
!pip install xgboost

In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
from joblib import Parallel, delayed
from datetime import time

# ===== Helper Functions for Indicator Calculations =====
def calculate_indicators(data):
    """
    Calculate VWAP, RSI, and Stochastic Oscillator for the given data.
    """
    # VWAP Calculation
    data['Typical Price'] = (data['high'] + data['low'] + data['close']) / 3
    data['Cumulative Vol * Price'] = (data['volume'] * data['Typical Price']).cumsum()
    data['Cumulative Volume'] = data['volume'].cumsum()
    data['VWAP'] = data['Cumulative Vol * Price'] / data['Cumulative Volume']

    # RSI Calculation
    delta = data['close'].diff()
    gain = delta.clip(lower=0).rolling(window=31).mean()
    loss = -delta.clip(upper=0).rolling(window=31).mean()
    data['RSI'] = 100 - (100 / (1 + gain / loss))

    # Stochastic Oscillator
    data['Low_Min'] = data['low'].rolling(window=8).min()
    data['High_Max'] = data['high'].rolling(window=8).max()
    data['%K'] = ((data['close'] - data['Low_Min']) / (data['High_Max'] - data['Low_Min'])) * 100
    data['%K'] = data['%K'].rolling(window=7).mean()  # Smoothing %K
    data['%D'] = data['%K'].rolling(window=3).mean()  # Smoothing %D

    # Drop NaN values due to rolling calculations
    data.dropna(inplace=True)

# ===== Trading Strategy =====
def apply_trading_strategy(
    ticker_data, capital=100000, stop_loss_pct=0.02, take_profit_pct=0.02, brokerage_pct=0.001
):
    """
    Apply a trading strategy based on indicators and generate buy/sell signals.
    """
    tradebook = []  # To store trade details
    position = 0  # Track current position (0 = no position, > 0 = long, < 0 = short)
    entry_price = 0
    entry_time = None
    cumulative_pnl = 0  # Cumulative PnL

    # Define trading times
    cutoff_time = time(11, 30)  # 11:30 AM
    end_time = time(15, 25)  # 15:25 PM

    ticker_data['date'] = pd.to_datetime(ticker_data['date'])

    # Trading Conditions
    buy_condition = (
        (ticker_data['RSI'] < 35)
        & (ticker_data['%K'] < 20)
        & (ticker_data['close'] > ticker_data['VWAP'])
    )
    short_condition = (
        (ticker_data['RSI'] > 70)
        & (ticker_data['%K'] > 70)
        & (ticker_data['close'] < ticker_data['VWAP'])
    )

    # Add tqdm to monitor the loop
    for i, row in tqdm(ticker_data.iterrows(), total=len(ticker_data), desc="Processing Trades"):
        # Skip trades after 11:30 AM
        if row['date'].time() > cutoff_time and position == 0:
            continue

        # Entry Logic
        if position == 0:  # No current position
            if buy_condition[i] and row['date'].time() <= cutoff_time:
                position = capital / row['close']
                entry_price = row['close']
                entry_time = row['date']
                tradebook.append({
                    'Ticker': ticker_data['Ticker'].iloc[0],
                    'Entry Time': entry_time,
                    'Entry Price': entry_price,
                    'Entry Shares': position,
                    'Exit Time': None,
                    'Exit Price': None,
                    'PNL': None,
                    'Position': 'Long',
                })
            elif short_condition[i] and row['date'].time() <= cutoff_time:
                position = -capital / row['close']
                entry_price = row['close']
                entry_time = row['date']
                tradebook.append({
                    'Ticker': ticker_data['Ticker'].iloc[0],
                    'Entry Time': entry_time,
                    'Entry Price': entry_price,
                    'Entry Shares': position,
                    'Exit Time': None,
                    'Exit Price': None,
                    'PNL': None,
                    'Position': 'Short',
                })

        # Exit Logic: Close position at 15:25 PM or based on conditions
        if position != 0:  # If there's a position
            exit_price = row['close']
            shares = abs(position)

            # Calculate Brokerage
            brokerage = (entry_price + exit_price) * shares * brokerage_pct

            # Calculate PnL
            pnl = (exit_price - entry_price) * shares - brokerage

            # Close position at 15:25 PM
            if row['date'].time() >= end_time:
                position = 0
                cumulative_pnl += pnl
                tradebook[-1].update({
                    'Exit Time': row['date'],
                    'Exit Price': exit_price,
                    'PNL': pnl,
                    'Cumulative PNL': cumulative_pnl,
                })

            # Exit Logic for Profit Taking or Stop Loss (before 15:25 PM)
            elif position > 0:  # Long Exit
                take_profit = entry_price * (1 + take_profit_pct)
                stop_loss = entry_price * (1 - stop_loss_pct)
                if row['close'] >= take_profit or row['close'] <= stop_loss:
                    position = 0
                    cumulative_pnl += pnl
                    tradebook[-1].update({
                        'Exit Time': row['date'],
                        'Exit Price': exit_price,
                        'PNL': pnl,
                        'Cumulative PNL': cumulative_pnl,
                    })

            elif position < 0:  # Short Exit
                take_profit = entry_price * (1 - take_profit_pct)
                stop_loss = entry_price * (1 + stop_loss_pct)
                if row['close'] <= take_profit or row['close'] >= stop_loss:
                    position = 0
                    cumulative_pnl += pnl
                    tradebook[-1].update({
                        'Exit Time': row['date'],
                        'Exit Price': exit_price,
                        'PNL': pnl,
                        'Cumulative PNL': cumulative_pnl,
                    })

    return pd.DataFrame(tradebook)

# ===== Machine Learning Model for Trade Profitability Prediction =====
def train_ml_model(data, tradebook):
    """
    Train an XGBoost model to predict trade profitability based on historical trades and indicators.
    """
    features = ['RSI', 'VWAP', '%K', '%D', 'volume', 'close']

    # Merge the data and tradebook
    merged_data = pd.merge(data, tradebook, left_on='date', right_on='Entry Time', how='inner')

    # Drop rows with missing values in features or PNL
    merged_data.dropna(subset=features + ['PNL'], inplace=True)

    # Extract features and target
    X = merged_data[features]
    y = (merged_data['PNL'] > 0).astype(int)  # Target: 1 if profit, 0 if loss

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train the XGBoost model
    model = XGBClassifier(random_state=42)
    model.fit(X_train, y_train)

    # Evaluate the model
    print("Model Accuracy:", accuracy_score(y_test, model.predict(X_test)))
    print(classification_report(y_test, model.predict(X_test)))

    return model

# ===== Live Trading Decision Based on ML Model =====
def live_trade_decision(data, model):
    """
    Make live trading decisions based on the trained ML model and real-time data.
    """
    calculate_indicators(data)  # Calculate indicators for live data
    features = ['RSI', 'VWAP', '%K', '%D', 'volume', 'close']

    # Get predicted probabilities of profitability
    data['Predicted_Profitability'] = model.predict_proba(data[features])[:, 1]

    # Filter trades with high predicted profitability (> 90%)
    high_prob_trades = data[data['Predicted_Profitability'] > 0.7]

    return high_prob_trades

# ===== Main Execution =====
def backtest_and_live_trade(data, tradebook):
    """
    Perform backtesting, training, and live trading decision-making.
    """
    # Step 1: Calculate indicators
    calculate_indicators(data)

    # Step 2: Apply the trading strategy
    tradebook = apply_trading_strategy(data)

    # Step 3: Train ML model
    ml_model = train_ml_model(data, tradebook)

    # Step 4: Make live trading decisions
    live_trades = live_trade_decision(data, ml_model)

    return pd.DataFrame(tradebook), live_trades

# Example Usage
tradebook, live_trades = backtest_and_live_trade(data, [])


In [ ]:
# Save the tradebook and trade decisions to CSV
tradebook.to_csv('/content/drive/MyDrive/tradebook.csv', index=False)
trade_decisions.to_csv('/content/drive/MyDrive/trade_decisions.csv', index=False)

In [ ]:
data

In [ ]:
tradebook_df

In [ ]:
import pandas as pd
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# Feature columns (indicators, volume, etc.)
features = ['RSI', 'VWAP', '%K', '%D', 'volume', 'close']

# Merge data and tradebook_df
merged_data = pd.merge(data, tradebook_df, left_on='date', right_on='Entry Time', how='inner')

# Drop rows with missing values in features or PNL
merged_data.dropna(subset=features + ['PNL'], inplace=True)

# Extract X (features) and y (target)
X = merged_data[features]
y = (merged_data['PNL'] > 0).astype(int)  # Target: 1 if profit, 0 if loss

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Train XGBoost model
model = XGBClassifier(random_state=42)
model.fit(X_train, y_train)

# Evaluate model
y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

# Add model predictions to data
merged_data['Predicted_Profitability'] = model.predict_proba(X)[:, 1]

# Filter trades with high probability of success
high_prob_trades = merged_data[merged_data['Predicted_Profitability'] > 0.9]
print(high_prob_trades)


In [ ]:
s = (high_prob_trades).to_csv('high_prob_trades.csv')

In [ ]:
def calculate_win_rate(tradebook_df):
    """
    Calculate the win rate from a tradebook DataFrame.

    Parameters:
    - tradebook_df (pd.DataFrame): A DataFrame containing trade details with a 'PNL' column.

    Returns:
    - float: Win rate as a percentage.
    """
    # Filter valid trades with PNL values
    valid_trades = tradebook_df.dropna(subset=['PNL'])

    # Count winning and total trades
    winning_trades = valid_trades[valid_trades['PNL'] > 0].shape[0]
    total_trades = valid_trades.shape[0]

    # Calculate win rate
    win_rate = (winning_trades / total_trades) * 100 if total_trades > 0 else 0.0

    print(f"Winning Trades: {winning_trades}")
    print(f"Total Trades: {total_trades}")
    print(f"Win Rate: {win_rate:.2f}%")
    return win_rate
# Example usage
win_rate = calculate_win_rate(tradebook_df)


In [ ]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from itertools import product
from joblib import Parallel, delayed  # For parallel processing

# ===== Helper Functions for Indicator Calculations =====
def calculate_indicators(data, rsi_window, stoch_k_window, stoch_d_window):
    """
    Precompute VWAP, RSI, and Stochastic Oscillator for the entire dataset.
    """
    data['Typical Price'] = (data['high'] + data['low'] + data['close']) / 3
    data['Cumulative Vol * Price'] = (data['volume'] * data['Typical Price']).cumsum()
    data['Cumulative Volume'] = data['volume'].cumsum()
    data['VWAP'] = data['Cumulative Vol * Price'] / data['Cumulative Volume']
    del data['Typical Price'], data['Cumulative Vol * Price'], data['Cumulative Volume']  # Free memory

    delta = data['close'].diff()
    gain = delta.clip(lower=0).rolling(window=rsi_window).mean()
    loss = -delta.clip(upper=0).rolling(window=rsi_window).mean()
    data['RSI'] = 100 - (100 / (1 + gain / loss))

    data['Low_Min'] = data['low'].rolling(window=stoch_k_window).min()
    data['High_Max'] = data['high'].rolling(window=stoch_k_window).max()
    data['%K'] = ((data['close'] - data['Low_Min']) / (data['High_Max'] - data['Low_Min'])) * 100
    data['%K'] = data['%K'].rolling(window=stoch_d_window).mean()
    data['%D'] = data['%K'].rolling(window=3).mean()
    data.drop(columns=['Low_Min', 'High_Max'], inplace=True)  # Free memory

    return data


# ===== Backtesting Function =====
def backtest_single_ticker(ticker_data, capital, stop_loss_pct, take_profit_pct,
                           rsi_buy_threshold, rsi_sell_threshold,
                           stoch_k_buy_threshold, stoch_k_sell_threshold,
                           cutoff_time, end_time, brokerage):
    """
    Backtest for a single ticker's data.
    """
    total_pnl = 0
    total_trades = 0
    winning_trades = 0
    position = 0
    entry_price = 0

    ticker_data = ticker_data.dropna(subset=['VWAP', 'RSI', '%K', '%D'])  # Clean data

    for _, row in ticker_data.iterrows():
        # Skip excluded day (default: Friday)
        if row['date'].weekday() == 4:
            continue

        buy_condition = (row['RSI'] < rsi_buy_threshold) and (row['%K'] < stoch_k_buy_threshold) and (row['close'] > row['VWAP'])
        short_condition = (row['RSI'] > rsi_sell_threshold) and (row['%K'] > stoch_k_sell_threshold) and (row['close'] < row['VWAP'])

        # Entry Logic
        if position == 0:
            if buy_condition and row['date'].time() <= cutoff_time:
                position = capital / row['close']
                entry_price = row['close']
            elif short_condition and row['date'].time() <= cutoff_time:
                position = -capital / row['close']
                entry_price = row['close']

        # Exit Logic
        if position != 0:
            exit_price = row['close']
            pnl = (exit_price - entry_price) * position - brokerage * capital

            if position > 0:  # Long Exit
                take_profit = entry_price * (1 + take_profit_pct)
                stop_loss = entry_price * (1 - stop_loss_pct)
                if row['close'] >= take_profit or row['close'] <= stop_loss or row['date'].time() == end_time:
                    total_pnl += pnl
                    total_trades += 1
                    if pnl > 0:
                        winning_trades += 1
                    position = 0

            elif position < 0:  # Short Exit
                take_profit = entry_price * (1 - take_profit_pct)
                stop_loss = entry_price * (1 + stop_loss_pct)
                if row['close'] <= take_profit or row['close'] >= stop_loss or row['date'].time() == end_time:
                    total_pnl += pnl
                    total_trades += 1
                    if pnl > 0:
                        winning_trades += 1
                    position = 0

    win_rate = (winning_trades / total_trades) * 100 if total_trades > 0 else 0
    return total_pnl, win_rate


def backtest_strategy(data, rsi_window, stoch_k_window, stoch_d_window, **params):
    """
    Backtest the strategy for all tickers in the dataset.
    """
    # Precompute indicators for the current set of parameters
    data = calculate_indicators(data.copy(), rsi_window, stoch_k_window, stoch_d_window)

    total_pnl = 0
    total_trades = 0
    winning_trades = 0

    # Iterate through tickers
    for ticker in data['Ticker'].unique():
        ticker_data = data[data['Ticker'] == ticker].copy()
        pnl, win_rate = backtest_single_ticker(ticker_data, **params)
        total_pnl += pnl
        total_trades += total_trades
        winning_trades += winning_trades

    win_rate = (winning_trades / total_trades) * 100 if total_trades > 0 else 0
    return total_pnl, win_rate


# ===== Parallelized Parameter Tuning =====
def run_backtest_for_params(params):
    """
    Wrapper function for parallel backtesting with specific parameters.
    """
    total_pnl, win_rate = backtest_strategy(data=data, **params)
    return {**params, 'Total PnL': total_pnl, 'Win Rate': win_rate}


# Parameter combinations
param_grid = {
    'rsi_window': [31, 41, 51],
    'stoch_k_window': [5, 7, 8, 9],
    'stoch_d_window': [5, 7],
    'capital': [100000],
    'stop_loss_pct': [0.02],
    'take_profit_pct': [0.02],
    'rsi_buy_threshold': [25, 35],
    'rsi_sell_threshold': [70, 75],
    'stoch_k_buy_threshold': [15, 20],
    'stoch_k_sell_threshold': [70, 75],
    'cutoff_time': [pd.to_datetime("11:30").time()],
    'end_time': [pd.to_datetime("15:25").time()],
    'brokerage': [0.001]
}
param_combinations = [dict(zip(param_grid.keys(), values)) for values in product(*param_grid.values())]

# Parallelize the backtesting process
results = Parallel(n_jobs=-1)(delayed(run_backtest_for_params)(params) for params in tqdm(param_combinations))

# Save results
results_df = pd.DataFrame(results)
results_df.to_csv('/content/drive/MyDrive/fine_tuning_results.csv', index=False)
